# Lab 02: Core ML & Retrieval Algorithms from Scratch (NumPy)

**Companion to [Module 13](../curriculum/05-production-and-practice/13_algorithms_from_scratch.md)** — Practice live-coding Linear Regression, Stable Softmax, K-Means, and BM25.

In [ ]:
import numpy as np

# 1. Numerically Stable Softmax & Cross-Entropy
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

probs = stable_softmax(np.array([[1000.0, 1001.0, 1002.0]]))
assert not np.isnan(probs).any() and np.isclose(probs.sum(), 1.0)
print('Stable probabilities:', np.round(probs, 4))

In [ ]:
# 2. Vectorized Linear Regression via Batch Gradient Descent
X = np.array([[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [4.0, 5.0]])
y = np.array([5.0, 8.0, 11.0, 14.0])  # y = 1*x1 + 2*x2

w = np.zeros(X.shape[1])
b = 0.0
for _ in range(2000):
    err = (X @ w + b) - y
    w -= 0.02 * (2.0 / len(X)) * (X.T @ err)
    b -= 0.02 * (2.0 / len(X)) * float(np.sum(err))

pred = float(np.array([5.0, 6.0]) @ w + b)
assert np.isclose(pred, 17.0, atol=1e-2)
print(f'Predicted: {pred:.2f} (Expected: 17.00)')

In [ ]:
# 3. O(N) Top-K Cosine Search using np.argpartition
def top_k_cosine(query: np.ndarray, corpus: np.ndarray, k: int = 2) -> np.ndarray:
    qn = max(float(np.linalg.norm(query)), 1e-12)
    cn = np.maximum(np.linalg.norm(corpus, axis=1), 1e-12)
    sims = (corpus @ query) / (cn * qn)
    idx = np.argpartition(-sims, kth=k - 1)[:k]
    return idx[np.argsort(-sims[idx])]

docs = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.95, 1.0]])
best = top_k_cosine(np.array([1.0, 1.0, 1.0]), docs, k=2)
assert best[0] == 2
print('Top-2 closest doc indices:', best)